# Find Optical Artifact-Only Units

Goal: flag Kilosort and BombCell units that are likely optical stimulation artifacts rather than neurons.

This notebook looks for units that:
- fire immediately after optical pulses aligned to neural time
- repeat across known 10-pulse light trains
- respond on light trials but not on no-light control events
- spend most of their spikes inside a narrow post-light window


In [83]:
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import ListedColormap

%matplotlib inline
%load_ext autoreload
%autoreload 2

import load_nwb
from load_nwb import nwb_loader
import nwb_data_prep as nwb_prep
import pca_data_prep as prep
import prep_data as data_prep

prep = importlib.reload(prep)
nwb_prep = importlib.reload(nwb_prep)
data_prep = importlib.reload(data_prep)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Load Session From `.env`


In [84]:
session_data_dic = prep.load_env()
session_data_dic


MOUSE loaded: Reach15




-- Behavioral Files --
BEHAVIORAL_FOLDER loaded: grant_reach15_swingDoor-christie
-- First Neuropixels File --
NP_FILE loaded: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
NWB_FILE loaded: Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01
DATE loaded: 20260129
SESSION loaded: session003
BOMBCELL loaded: bombcell_batch_20260305_1130
PROBE_A_CH_CONFIG loaded: probeA_SIM_IP__Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01.json
PROBE_C_CH_CONFIG loaded: probeC_MoP__Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01.json
PROBE_D_CH_CONFIG loaded: probeD_VaL__Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01.json


-- Second Neuropixels File --
NP_FILE loaded: Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32
NWB_FILE loaded: NA
DATE_01 loaded: 20260129
SESSION_01 loaded: session004
BOMBCELL_01 loaded: NA
PROBE_A_CH_CONFIG_01 loaded: NA
PROBE_C_CH_CONFIG_01 loaded: NA
PROBE_D_CH_CONFIG_01 loa

{'MOUSE': 'Reach15',
 'BEHAVIORAL_FOLDER': 'grant_reach15_swingDoor-christie',
 'NP_FILE': 'Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01',
 'DATE': '20260129',
 'SESSION': 'session003',
 'BOMBCELL': 'bombcell_batch_20260305_1130',
 'NWB_FILE': 'Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01',
 'PROBE_A_CH_CONFIG': 'probeA_SIM_IP__Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01.json',
 'PROBE_C_CH_CONFIG': 'probeC_MoP__Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01.json',
 'PROBE_D_CH_CONFIG': 'probeD_VaL__Reach15_20260129_session003_NP_Recording_2026-01-29_14-30-01.json',
 'NP_FILE_01': 'Reach15_20260129_session004_NP_Recording_02_2026-01-29_16-50-32',
 'DATE_01': '20260129',
 'SESSION_01': 'session004',
 'BOMBCELL_01': 'NA',
 'NWB_FILE_01': 'NA',
 'PROBE_A_CH_CONFIG_01': 'NA',
 'PROBE_C_CH_CONFIG_01': 'NA',
 'PROBE_D_CH_CONFIG_01': 'NA',
 'NP_FILE_02': 'Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00',
 'DA

In [85]:
SESSION_TO_ANALYZE = 3

MOUSE, BEHAVIORAL_FOLDER, PROBE_A_CH_CONFIG, PROBE_C_CH_CONFIG, PROBE_D_CH_CONFIG, NP_FILE, NWB_FILE, DATE, SESSION, BOMBCELL = prep.session_to_analyze(
    session_data_dic['MOUSE'],
    session_data_dic['BEHAVIORAL_FOLDER'],
    session_data_dic['NP_FILE'],
    session_data_dic['NWB_FILE'],
    session_data_dic['DATE'],
    session_data_dic['SESSION'],
    session_data_dic['BOMBCELL'],
    session_data_dic['PROBE_A_CH_CONFIG'],
    session_data_dic['PROBE_C_CH_CONFIG'],
    session_data_dic['PROBE_D_CH_CONFIG'],
    session_data_dic['NP_FILE_01'],
    session_data_dic['NWB_FILE_01'],
    session_data_dic['DATE_01'],
    session_data_dic['SESSION_01'],
    session_data_dic['BOMBCELL_01'],
    session_data_dic['PROBE_A_CH_CONFIG_01'],
    session_data_dic['PROBE_C_CH_CONFIG_01'],
    session_data_dic['PROBE_D_CH_CONFIG_01'],
    session_data_dic['NP_FILE_02'],
    session_data_dic['NWB_FILE_02'],
    session_data_dic['DATE_02'],
    session_data_dic['SESSION_02'],
    session_data_dic['BOMBCELL_02'],
    session_data_dic['PROBE_A_CH_CONFIG_02'],
    session_data_dic['PROBE_C_CH_CONFIG_02'],
    session_data_dic['PROBE_D_CH_CONFIG_02'],
    session_selection=SESSION_TO_ANALYZE,
)

PROBES = ['A', 'B', 'C', 'D', 'E', 'F']
paths_dic = prep.setup_paths_and_verify(PROBES, NWB_FILE, NP_FILE, DATE, SESSION, BEHAVIORAL_FOLDER, BOMBCELL)
paths_dic



SESSION SELECTION:

NP_FILE: Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00
NWB_FILE: Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00
DATE: 20260201
SESSION: session007
BEHAVIORAL_FOLDER: grant_reach15_swingDoor-christie
BOMBCELL: bombcell_batch_20260304_1536
PROBE_A_CH_CONFIG: probeA_SIM_IP__Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00.json
PROBE_C_CH_CONFIG: probeC_MoP__Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00.json
PROBE_D_CH_CONFIG: probeD_VaL__Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00.json
✅ Found bombcell data for probe A at H:\Grant\Neuropixel_Analysis\BOMBCELL\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell_batch_20260304_1536\kilosort4_A
✅ Found bombcell data for probe B at H:\Grant\Neuropixel_Analysis\BOMBCELL\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00\bombcell_batch_20260304_1536\kilosort4_B
✅

{'NP_ROOT_DIR': WindowsPath('H:/Grant/Neuropixels/Kilosort_Recordings/Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00'),
 'NWB_PATH': WindowsPath('H:/Grant/Neuropixel_Analysis/NWB/Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00'),
 'BOMBCELL_ROOT_FOR_AUTO_BUILD': WindowsPath('H:/Grant/Neuropixel_Analysis/BOMBCELL/Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00/bombcell_batch_20260304_1536'),
 'baseline_trials_index_path': 'G:\\Grant\\behavior_data\\DLC_net\\grant_reach15_swingDoor-christie\\videos\\20260201\\christielab\\session007\\20260201_christielab_session007_baseline_trial_numbers_tone2_aligned.npy',
 'washout_trials_index_path': 'G:\\Grant\\behavior_data\\DLC_net\\grant_reach15_swingDoor-christie\\videos\\20260201\\christielab\\session007\\20260201_christielab_session007_washout_trial_numbers_tone2_aligned.npy',
 'optoicalStim_trials_index_path': 'G:\\Grant\\behavior_data\\DLC_net\\grant_reach15_swingDoor-christie

## Load Units and Event Arrays


In [86]:
def combine_probe_tables(merged_dic):
    frames = []
    for probe, df_probe in merged_dic.items():
        if df_probe is None or len(df_probe) == 0:
            continue
        frame = df_probe.copy()
        if 'probe' not in frame.columns:
            frame['probe'] = probe
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True, sort=False)


mouse_nwb = paths_dic['NWB_PATH']
baseline_trials_index_path = paths_dic['baseline_trials_index_path']
washout_trials_index_path = paths_dic['washout_trials_index_path']
optoicalStim_trials_index_path = paths_dic['optoicalStim_trials_index_path']
BOMBCELL_ROOT_FOR_AUTO_BUILD = paths_dic['BOMBCELL_ROOT_FOR_AUTO_BUILD']

mouse_id = nwb_loader(mouse_nwb)
df_stim, df_units_nwb = mouse_id.verify_nwb_data()

PROCESSED_BUNDLE_SESSION_DIR = Path.cwd().resolve() / 'processed_data' / NP_FILE
PROCESSED_BUNDLE_SESSION_DIR.mkdir(parents=True, exist_ok=True)

bundle, merged_dic, stim_df, _, pca_event_meta_bundle, extras, meta, PROCESSED_BUNDLE_DIR = nwb_prep.load_or_build_processed_bundle(
    processed_bundle_dir=PROCESSED_BUNDLE_SESSION_DIR,
    nwb_path_for_auto_build=mouse_nwb,
    bombcell_root_for_auto_build=BOMBCELL_ROOT_FOR_AUTO_BUILD,
    use_bombcell_if_available=True,
    auto_build_bundle_if_missing=True,
    auto_rebuild_if_bombcell_missing=True,
    required_filenames=('merged_dic.pkl', 'stim_df.pkl', 'pca_event_meta.pkl'),
    verbose=True,
)

units_df = combine_probe_tables(merged_dic)
if units_df.empty:
    units_df = df_units_nwb.copy()

if 'cluster_id' not in units_df.columns:
    if 'id' in units_df.columns:
        units_df = units_df.rename(columns={'id': 'cluster_id'})
    else:
        units_df = units_df.reset_index().rename(columns={'index': 'cluster_id'})

if 'spike_times' not in units_df.columns and 'times' in units_df.columns:
    units_df['spike_times'] = units_df['times']

baseline_trials_idx = np.load(baseline_trials_index_path, allow_pickle=True)
washout_trials_idx = np.load(washout_trials_index_path, allow_pickle=True)
optoicalStim_trials_idx = np.load(optoicalStim_trials_index_path, allow_pickle=True)

tone1_start_times, tone2_start_times, frame_events_start_times, stimROI_start_times, optical_start_times, all_stimROI_triggers_start_times = data_prep.extract_start_times(df_stim)

(
    pca_event_meta,
    tone1_start_times,
    tone2_start_times,
    stimROI_start_times,
    optical_start_times,
    all_stimROI_triggers_start_times,
    baseline_reachInit_stimROI_start_times,
    stimulation_reachInit_stimROI_start_times,
    washout_reachInit_stimROI_start_times,
) = prep.build_pca_event_meta_and_event_times(
    stim_df=stim_df,
    baseline_trials_idx=baseline_trials_idx,
    optoicalStim_trials_idx=optoicalStim_trials_idx,
    washout_trials_idx=washout_trials_idx,
)

opto_tagging_timestamps = df_stim.loc[
    df_stim['stimulus'] == 'opto_tagging_timestamps',
    'start_time',
].to_numpy(dtype=float)

known_light_train_start_times = np.sort(
    np.concatenate([stimulation_reachInit_stimROI_start_times, opto_tagging_timestamps]).astype(float)
)
known_no_light_start_times = np.sort(
    np.concatenate([tone1_start_times, washout_reachInit_stimROI_start_times]).astype(float)
)

light_train_start_meta = pd.DataFrame(
    {
        'train_start_time': np.concatenate([stimulation_reachInit_stimROI_start_times, opto_tagging_timestamps]).astype(float),
        'train_source': (['closed_loop'] * len(stimulation_reachInit_stimROI_start_times))
        + (['opto_tagging'] * len(opto_tagging_timestamps)),
    }
).sort_values('train_start_time').reset_index(drop=True)

print(f'Units table rows: {len(units_df)}')
print(f'opto_tagging_timestamps: {len(opto_tagging_timestamps)}')
print(f'stimulation_reachInit_stimROI_timestamps: {len(stimulation_reachInit_stimROI_start_times)}')
print(f'Known light train starts: {len(known_light_train_start_times)}')
print(f'Known no-light starts: {len(known_no_light_start_times)}')
print(f'Raw optical pulses in recording: {len(optical_start_times)}')
units_df.head()


loaded NWB from: H:\Grant\Neuropixel_Analysis\NWB\Reach15_20260201_session007_NP_Recording_Number02_2026-02-01_18-25-00

===== Total units Per Probe ====
A:  554
B:  1316
C:  355
D:  820
E:  864
F:  691

 ======= Unique stimulus types ==========   : 
 ['tone1_timestamps' 'tone2_timestamps' 'stimROI_timestamps'
 'frame_events_timestamp' 'optical_timestamps'
 'reachInit_stimROI_timestamps' 'opto_tagging_timestamps'
 'baseline_reachInit_stimROI_timestamps'
 'stimulation_reachInit_stimROI_timestamps'
 'washout_reachInit_stimROI_timestamps']

===== Total Timestamps Per Event ====
tone1_timestamps:  376
tone2_timestamps:  270
stimROI_timestamps:  130
frame_events_timestamp:  1349818
optical_timestamps:  1900
reachInit_stimROI_timestamps:  270
opto_tagging_timestamps:  60
baseline_reachInit_stimROI_timestamps:  20
stimulation_reachInit_stimROI_timestamps:  130
washout_reachInit_stimROI_timestamps:  120


Loaded bundle: C:\Users\user\Documents\github\NWB_and_Reaching_analysis\mice\Reach15\anal

,depth,xpos,ypos,label,KSlabel,KSamplitude,KScontamination,probe,bc_ROI_x,Brain_Region_x,bc_unitType,spike_times,cluster_id,nSpikes,maxDriftEstimate,maxChannels,bc_label,bc_ROI_y,Brain_Region_y,in_brainRegion,brain_region
0,3120.0,277.0,30.0,0,2,9.5,14.5,A,IN_ROI,IP,NOISE,"[9938.427594575005, 9938.443329123113, 9938.44...",0,14579.0,29.717560,52,NOISE,IN_ROI,IP,NaN,NaN
1,3120.0,277.0,30.0,0,2,10.3,17.2,A,IN_ROI,IP,NOISE,"[9938.363989665882, 9938.408826460065, 9938.42...",1,35469.0,93.164818,52,NOISE,IN_ROI,IP,NaN,NaN
2,3120.0,59.0,30.0,0,1,9.0,0.0,A,IN_ROI,IP,NOISE,"[9938.135738714089, 9939.27045963741, 9940.801...",2,176.0,4.297394,5,NOISE,IN_ROI,IP,NaN,NaN
3,2775.0,27.0,375.0,0,2,8.5,0.0,A,IN_ROI,IP,NOISE,"[9938.897430842511, 9940.381573391209, 9941.04...",3,386.0,403.393829,98,NOISE,IN_ROI,IP,NaN,NaN
4,2745.0,59.0,405.0,0,2,9.1,7.4,A,IN_ROI,IP,NOISE,"[9938.29451763671, 9938.519134975282, 9938.609...",4,10164.0,614.671204,103,NOISE,IN_ROI,IP,NaN,NaN


## Detection Parameters and Helpers

`optical_start_times` are split into closed-loop and opto-tagging blocks, chunked into 10-pulse trains, and then each unit is scored for pulse-locking and selectivity to light windows.

In [87]:
EXPECTED_PULSES_PER_TRAIN = 10
EXPECTED_ISI_S = 0.010
TRAIN_ISI_TOL_S = 0.003
RESPONSE_WINDOW_S = (0.000, 0.004)
BASELINE_WINDOW_S = (-0.020, -0.002)
EXCLUSIVE_WINDOW_S = (0.000, 0.006)
MIN_PULSE_RESPONSE_FRAC = 0.80
MIN_NEAR_FULL_TRAIN_FRAC = 0.80
MIN_EXCLUSIVE_SPIKE_FRAC = 0.80
MAX_LATENCY_JITTER_MS = 0.75
MIN_RESPONSE_BASELINE_RATIO = 8.0
MAX_CONTROL_START_RESPONSE_FRAC = 0.20
MIN_LIGHT_CONTROL_DIFF = 0.60


def chunk_pulse_trains(pulse_times, pulses_per_train=10):
    pulse_times = np.sort(np.asarray(pulse_times, dtype=float))
    if pulse_times.size == 0:
        return []

    n_full_trains = pulse_times.size // pulses_per_train
    return [
        pulse_times[idx * pulses_per_train : (idx + 1) * pulses_per_train]
        for idx in range(n_full_trains)
    ]


def summarize_pulse_trains(
    pulse_trains,
    known_start_times,
    train_source,
    expected_pulses_per_train=10,
    expected_isi_s=0.010,
):
    summary_columns = [
        'train_idx',
        'train_source',
        'known_train_start_time',
        'pulse_train_start_time',
        'start_time_delta_s',
        'candidate_group_size',
        'window_start_idx',
        'n_pulses',
        'median_isi_s',
        'max_isi_error_s',
        'matches_template',
    ]
    known_start_times = np.asarray(known_start_times, dtype=float)
    summary_records = []

    for idx, train in enumerate(pulse_trains):
        train = np.asarray(train, dtype=float)
        isi = np.diff(train)
        pulse_train_start_time = float(train[0]) if len(train) else np.nan
        known_train_start_time = float(known_start_times[idx]) if idx < len(known_start_times) else np.nan
        max_isi_error = float(np.max(np.abs(isi - expected_isi_s))) if len(isi) else np.nan

        summary_records.append({
            'train_idx': idx,
            'train_source': train_source,
            'known_train_start_time': known_train_start_time,
            'pulse_train_start_time': pulse_train_start_time,
            'start_time_delta_s': (
                float(pulse_train_start_time - known_train_start_time)
                if np.isfinite(pulse_train_start_time) and np.isfinite(known_train_start_time)
                else np.nan
            ),
            'candidate_group_size': int(len(train)),
            'window_start_idx': 0,
            'n_pulses': int(len(train)),
            'median_isi_s': float(np.median(isi)) if len(isi) else np.nan,
            'max_isi_error_s': max_isi_error,
            'matches_template': bool(len(train) == expected_pulses_per_train),
        })

    return pd.DataFrame(summary_records, columns=summary_columns)


def count_spikes_in_windows(spike_times, event_times, window_s):
    spike_times = np.asarray(spike_times, dtype=float)
    event_times = np.asarray(event_times, dtype=float)
    if spike_times.size == 0 or event_times.size == 0:
        return np.zeros(len(event_times), dtype=int)

    counts = np.zeros(len(event_times), dtype=int)
    start_offset, end_offset = window_s
    for idx, event_time in enumerate(event_times):
        left = np.searchsorted(spike_times, event_time + start_offset, side='left')
        right = np.searchsorted(spike_times, event_time + end_offset, side='left')
        counts[idx] = right - left
    return counts


def first_spike_latencies(spike_times, event_times, window_s):
    spike_times = np.asarray(spike_times, dtype=float)
    event_times = np.asarray(event_times, dtype=float)
    latencies = np.full(len(event_times), np.nan, dtype=float)
    if spike_times.size == 0 or event_times.size == 0:
        return latencies

    start_offset, end_offset = window_s
    for idx, event_time in enumerate(event_times):
        left = np.searchsorted(spike_times, event_time + start_offset, side='left')
        right = np.searchsorted(spike_times, event_time + end_offset, side='left')
        if right > left:
            latencies[idx] = spike_times[left] - event_time
    return latencies


def score_artifact_units(units_df, pulse_trains, light_train_start_times, control_start_times):
    records = []
    light_train_start_times = np.asarray(light_train_start_times, dtype=float)
    control_start_times = np.asarray(control_start_times, dtype=float)

    all_light_pulses = np.concatenate(pulse_trains).astype(float) if pulse_trains else np.array([], dtype=float)

    for _, row in units_df.iterrows():
        spike_times = np.asarray(row['spike_times'], dtype=float)

        pulse_counts = count_spikes_in_windows(spike_times, all_light_pulses, RESPONSE_WINDOW_S)
        pulse_hits = pulse_counts > 0
        pulse_response_fraction = float(pulse_hits.mean()) if len(pulse_hits) else np.nan

        train_pulse_hit_counts = np.array([
            (count_spikes_in_windows(spike_times, train, RESPONSE_WINDOW_S) > 0).sum()
            for train in pulse_trains
        ], dtype=float) if pulse_trains else np.array([], dtype=float)
        train_response_fraction = float((train_pulse_hit_counts > 0).mean()) if len(train_pulse_hit_counts) else np.nan
        near_full_train_fraction = float((train_pulse_hit_counts >= (EXPECTED_PULSES_PER_TRAIN - 1)).mean()) if len(train_pulse_hit_counts) else np.nan

        latencies = first_spike_latencies(spike_times, all_light_pulses, RESPONSE_WINDOW_S)
        valid_latencies = latencies[np.isfinite(latencies)]
        latency_jitter_ms = float(np.std(valid_latencies) * 1000.0) if len(valid_latencies) > 1 else np.nan

        baseline_counts = count_spikes_in_windows(spike_times, all_light_pulses, BASELINE_WINDOW_S)
        response_rate_hz = float(np.mean(pulse_counts) / (RESPONSE_WINDOW_S[1] - RESPONSE_WINDOW_S[0])) if len(pulse_counts) else np.nan
        baseline_rate_hz = float(np.mean(baseline_counts) / (BASELINE_WINDOW_S[1] - BASELINE_WINDOW_S[0])) if len(baseline_counts) else np.nan
        response_baseline_ratio = float(response_rate_hz / max(baseline_rate_hz, 1e-6)) if np.isfinite(response_rate_hz) and np.isfinite(baseline_rate_hz) else np.nan

        light_start_counts = count_spikes_in_windows(spike_times, light_train_start_times, EXCLUSIVE_WINDOW_S)
        control_start_counts = count_spikes_in_windows(spike_times, control_start_times, EXCLUSIVE_WINDOW_S)
        light_train_start_response_fraction = float((light_start_counts > 0).mean()) if len(light_start_counts) else np.nan
        control_start_response_fraction = float((control_start_counts > 0).mean()) if len(control_start_counts) else np.nan
        light_control_response_diff = (
            float(light_train_start_response_fraction - control_start_response_fraction)
            if np.isfinite(light_train_start_response_fraction) and np.isfinite(control_start_response_fraction)
            else np.nan
        )

        exclusive_counts = count_spikes_in_windows(spike_times, all_light_pulses, EXCLUSIVE_WINDOW_S)
        spike_exclusive_to_light_fraction = float(exclusive_counts.sum() / len(spike_times)) if len(spike_times) else 0.0

        artifact_score = 0
        artifact_score += int(np.isfinite(pulse_response_fraction) and pulse_response_fraction >= MIN_PULSE_RESPONSE_FRAC)
        artifact_score += int(np.isfinite(near_full_train_fraction) and near_full_train_fraction >= MIN_NEAR_FULL_TRAIN_FRAC)
        artifact_score += int(np.isfinite(spike_exclusive_to_light_fraction) and spike_exclusive_to_light_fraction >= MIN_EXCLUSIVE_SPIKE_FRAC)
        artifact_score += int(np.isfinite(latency_jitter_ms) and latency_jitter_ms <= MAX_LATENCY_JITTER_MS)
        artifact_score += int(np.isfinite(response_baseline_ratio) and response_baseline_ratio >= MIN_RESPONSE_BASELINE_RATIO)
        artifact_score += int(np.isfinite(control_start_response_fraction) and control_start_response_fraction <= MAX_CONTROL_START_RESPONSE_FRAC)
        artifact_score += int(np.isfinite(light_control_response_diff) and light_control_response_diff >= MIN_LIGHT_CONTROL_DIFF)

        artifact_flag = bool(
            artifact_score >= 5
            and np.isfinite(pulse_response_fraction)
            and pulse_response_fraction >= MIN_PULSE_RESPONSE_FRAC
            and np.isfinite(near_full_train_fraction)
            and near_full_train_fraction >= MIN_NEAR_FULL_TRAIN_FRAC
            and np.isfinite(light_control_response_diff)
            and light_control_response_diff >= MIN_LIGHT_CONTROL_DIFF
        )

        records.append({
            'probe': row.get('probe', 'unknown'),
            'cluster_id': int(row['cluster_id']),
            'KSlabel': row.get('KSlabel', np.nan),
            'bc_label': row.get('bc_label', np.nan),
            'brain_region': row.get('brain_region', np.nan),
            'n_spikes': int(len(spike_times)),
            'artifact_score': artifact_score,
            'artifact_flag': artifact_flag,
            'pulse_response_fraction': pulse_response_fraction,
            'train_response_fraction': train_response_fraction,
            'near_full_train_fraction': near_full_train_fraction,
            'light_train_start_response_fraction': light_train_start_response_fraction,
            'control_start_response_fraction': control_start_response_fraction,
            'light_control_response_diff': light_control_response_diff,
            'latency_jitter_ms': latency_jitter_ms,
            'response_rate_hz': response_rate_hz,
            'baseline_rate_hz': baseline_rate_hz,
            'response_baseline_ratio': response_baseline_ratio,
            'spike_exclusive_to_light_fraction': spike_exclusive_to_light_fraction,
        })

    return pd.DataFrame(records).sort_values(
        ['artifact_flag', 'artifact_score', 'pulse_response_fraction', 'near_full_train_fraction'],
        ascending=[False, False, False, False],
    ).reset_index(drop=True)


def plot_pulse_train_structure(pulse_trains, max_trains=30, expected_isi_s=0.010):
    pulse_trains = [np.asarray(train, dtype=float) for train in pulse_trains if len(train)]
    if not pulse_trains:
        print('No pulse trains available for plotting.')
        return

    n_show = min(max_trains, len(pulse_trains))
    trains_to_show = pulse_trains[:n_show]
    rel_trains = [train - train[0] for train in trains_to_show]
    pulse_counts = np.asarray([len(train) for train in pulse_trains], dtype=int)
    start_times = np.asarray([train[0] for train in pulse_trains], dtype=float)
    all_isis_ms = np.concatenate([
        np.diff(train) * 1000.0
        for train in pulse_trains
        if len(train) > 1
    ]) if any(len(train) > 1 for train in pulse_trains) else np.array([], dtype=float)

    print(f'Total pulse trains: {len(pulse_trains)}')
    print(f'Pulse count per train: min={pulse_counts.min()}, median={np.median(pulse_counts):.0f}, max={pulse_counts.max()}')
    if len(start_times) > 1:
        print(f'Median gap between train starts: {np.median(np.diff(start_times)):.3f} s')

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), constrained_layout=True)

    axes[0].eventplot(rel_trains, orientation='horizontal', colors='black', linelengths=0.8)
    axes[0].set_ylabel('Train')
    axes[0].set_xlabel('Time from train start (s)')
    axes[0].set_title(f'First {n_show} pulse trains aligned to train start')

    pulse_count_summary = pd.Series(pulse_counts).value_counts().sort_index()
    axes[1].bar(pulse_count_summary.index.astype(str), pulse_count_summary.values, color='black')
    axes[1].set_ylabel('N trains')
    axes[1].set_xlabel('Pulses per train')
    axes[1].set_title('Pulse count per train')

    if len(all_isis_ms):
        axes[2].hist(all_isis_ms, bins=30, color='black')
        axes[2].axvline(expected_isi_s * 1000.0, color='tab:blue', linestyle='--', linewidth=1)
    axes[2].set_xlabel('Inter-pulse interval (ms)')
    axes[2].set_ylabel('Count')
    axes[2].set_title('All inter-pulse intervals')

    plt.show()


def plot_artifact_candidate(unit_row, pulse_trains, binsize_s=0.0005):
    spike_times = np.asarray(unit_row['spike_times'], dtype=float)
    n_trains = len(pulse_trains)
    if n_trains == 0:
        print('No pulse trains available for plotting.')
        return

    fig, axes = plt.subplots(3, 1, figsize=(10, 8), constrained_layout=True)

    for train_idx, train in enumerate(pulse_trains):
        rel_spike_times = []
        rel_event_times = train - train[0]
        for pulse_time in train:
            left = np.searchsorted(spike_times, pulse_time - 0.002, side='left')
            right = np.searchsorted(spike_times, pulse_time + 0.008, side='left')
            rel_spike_times.extend(spike_times[left:right] - train[0])
        if rel_spike_times:
            axes[0].scatter(rel_spike_times, np.full(len(rel_spike_times), train_idx), s=4, color='black')
        axes[0].vlines(rel_event_times, train_idx - 0.35, train_idx + 0.35, color='tab:blue', linewidth=0.8)

    axes[0].set_ylabel('Train')
    axes[0].set_title(f"Probe {unit_row.get('probe', 'unknown')} cluster {int(unit_row['cluster_id'])}")

    window_start = -0.002
    window_end = 0.008
    bins = np.arange(window_start, window_end + binsize_s, binsize_s)
    all_rel = []
    hit_matrix = np.zeros((n_trains, EXPECTED_PULSES_PER_TRAIN), dtype=int)

    for train_idx, train in enumerate(pulse_trains):
        for pulse_idx, pulse_time in enumerate(train):
            left = np.searchsorted(spike_times, pulse_time + window_start, side='left')
            right = np.searchsorted(spike_times, pulse_time + window_end, side='left')
            rel = spike_times[left:right] - pulse_time
            all_rel.extend(rel.tolist())
            hit_matrix[train_idx, pulse_idx] = int(np.any((rel >= RESPONSE_WINDOW_S[0]) & (rel < RESPONSE_WINDOW_S[1])))

    if all_rel:
        hist, edges = np.histogram(all_rel, bins=bins)
        rate = hist / max(n_trains * EXPECTED_PULSES_PER_TRAIN * binsize_s, 1e-9)
        axes[1].bar(edges[:-1], rate, width=np.diff(edges), align='edge', color='black')
    axes[1].axvspan(RESPONSE_WINDOW_S[0], RESPONSE_WINDOW_S[1], color='tab:blue', alpha=0.2)
    axes[1].set_ylabel('Hz')
    axes[1].set_title('Per-pulse PSTH')

    im = axes[2].imshow(hit_matrix, aspect='auto', interpolation='nearest', cmap='Greys', vmin=0, vmax=1)
    axes[2].set_xlabel('Pulse in train')
    axes[2].set_ylabel('Train')
    axes[2].set_title('Pulse hit map')
    fig.colorbar(im, ax=axes[2], shrink=0.8, label='Spike in response window')

    plt.show()


## Build Light Pulse Trains

These trains are built by splitting the verified closed-loop and opto-tagging pulse blocks and chunking them into 10-pulse trains.

In [88]:
(
    _,
    _,
    _,
    _,
    _,
    _,
    _,
    _,
    _,
    _,
    opto_closed_loop_start_timestamps,
    opto_tag_start_timestamps,
    _,
    first_opto_tagging_timestamp_per_trial,
    first_optical_pulse_per_closed_loop,
) = data_prep.seperate_closedLoop_optoTagging(
    optical_start_times=optical_start_times,
    tone2_start_times=tone2_start_times,
    frame_events_start_times=frame_events_start_times,
    total_opto_tagging_events=len(opto_tagging_timestamps),
    pulses_per_event=EXPECTED_PULSES_PER_TRAIN,
)

closed_loop_pulse_trains = chunk_pulse_trains(
    opto_closed_loop_start_timestamps,
    pulses_per_train=EXPECTED_PULSES_PER_TRAIN,
)
opto_tagging_pulse_trains = chunk_pulse_trains(
    opto_tag_start_timestamps,
    pulses_per_train=EXPECTED_PULSES_PER_TRAIN,
)

pulse_trains_to_use = closed_loop_pulse_trains + opto_tagging_pulse_trains
pulse_train_start_times = np.asarray([train[0] for train in pulse_trains_to_use], dtype=float)

pulse_train_summary = pd.concat(
    [
        summarize_pulse_trains(
            closed_loop_pulse_trains,
            known_start_times=stimulation_reachInit_stimROI_start_times,
            train_source='closed_loop',
            expected_pulses_per_train=EXPECTED_PULSES_PER_TRAIN,
            expected_isi_s=EXPECTED_ISI_S,
        ),
        summarize_pulse_trains(
            opto_tagging_pulse_trains,
            known_start_times=opto_tagging_timestamps,
            train_source='opto_tagging',
            expected_pulses_per_train=EXPECTED_PULSES_PER_TRAIN,
            expected_isi_s=EXPECTED_ISI_S,
        ),
    ],
    ignore_index=True,
    sort=False,
)

closed_loop_leftover_pulses = len(opto_closed_loop_start_timestamps) % EXPECTED_PULSES_PER_TRAIN
opto_tagging_leftover_pulses = len(opto_tag_start_timestamps) % EXPECTED_PULSES_PER_TRAIN

print(f'Closed-loop train starts: {len(stimulation_reachInit_stimROI_start_times)}')
print(f'Opto-tagging train starts: {len(opto_tagging_timestamps)}')
print(f'Total known light train starts: {len(light_train_start_meta)}')
print(f'Raw optical pulses in recording: {len(optical_start_times)}')
print(f'Closed-loop pulse trains: {len(closed_loop_pulse_trains)}')
print(f'Opto-tagging pulse trains: {len(opto_tagging_pulse_trains)}')
print(f'Total pulse trains to score: {len(pulse_trains_to_use)}')
print(f'Pulse train start times: {len(pulse_train_start_times)}')
print(f'Closed-loop leftover pulses: {closed_loop_leftover_pulses}')
print(f'Opto-tagging leftover pulses: {opto_tagging_leftover_pulses}')
print(f'No-light control starts: {len(known_no_light_start_times)}')
print('\nKnown light train sources:')
display(light_train_start_meta['train_source'].value_counts())
print('\nComplete 10-pulse train summary:')
display(pulse_train_summary.groupby(['train_source', 'matches_template']).size().rename('n_trains'))
print('\nPulse count per train:')
display(pulse_train_summary['n_pulses'].value_counts().sort_index())
print('\nStart time delta summary (pulse train start - known train start):')
display(pulse_train_summary.groupby('train_source')['start_time_delta_s'].describe())
pulse_train_summary.head(25)


Total optical start times: 1900
Total opto-tagging events: 60
Total opto-tagging pulses per event: 10
Last tone2 time: 18785.354166666668
Opto-tagging start time: 18840.179166666665, end time: 18958.665
Start of opto-tagging index: 1300

Estimated Behavioral Video duration (min): 149.98
Final behavioral Video Frame start_time: 19028.83

Total Closed Loop optical pulses: 1300
Total Opto-tagging optical pulses: 600


 ------- SUCCESS -------
 Optical Pulses have been seperated into closed loop and opto-tagging groups. And it has been verified that the first opto-tagging timestamp occurs after the last closed loop timestamp. 


Closed-loop train starts: 130
Opto-tagging train starts: 60
Total known light train starts: 190
Raw optical pulses in recording: 1900
Closed-loop pulse trains: 130
Opto-tagging pulse trains: 60
Total pulse trains to score: 190
Pulse train start times: 190
Closed-loop leftover pulses: 0
Opto-tagging leftover pulses: 0
No-light control starts: 496

Known light train 

closed_loop     130
opto_tagging     60
Name: train_source, dtype: int64


Complete 10-pulse train summary:


train_source  matches_template
closed_loop   True                130
opto_tagging  True                 60
Name: n_trains, dtype: int64


Pulse count per train:


10    190
Name: n_pulses, dtype: int64


Start time delta summary (pulse train start - known train start):


,count,mean,std,min,25%,50%,75%,max
train_source,,,,,,,,
closed_loop,130.0,0.000001,0.000006,0.0,0.0,0.0,0.0,0.000033
opto_tagging,60.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000


,train_idx,train_source,known_train_start_time,pulse_train_start_time,start_time_delta_s,candidate_group_size,window_start_idx,n_pulses,median_isi_s,max_isi_error_s,matches_template
0,0,closed_loop,10919.650067,10919.650067,0.0,10,0,10,0.01,0.005,True
1,1,closed_loop,10941.260433,10941.260433,0.0,10,0,10,0.01,0.005,True
2,2,closed_loop,10961.230467,10961.230467,0.0,10,0,10,0.01,0.005,True
3,3,closed_loop,11002.564267,11002.564267,0.0,10,0,10,0.01,0.005,True
4,4,closed_loop,11024.054767,11024.054767,0.0,10,0,10,0.01,0.005,True
5,5,closed_loop,11045.184867,11045.184867,0.0,10,0,10,0.01,0.005,True
6,6,closed_loop,11072.922900,11072.922900,0.0,10,0,10,0.01,0.005,True
7,7,closed_loop,11100.381000,11100.381000,0.0,10,0,10,0.01,0.005,True
8,8,closed_loop,11144.268633,11144.268633,0.0,10,0,10,0.01,0.005,True
9,9,closed_loop,11166.265933,11166.265933,0.0,10,0,10,0.01,0.005,True


## Quick Pulse Train Check

Use this plot to quickly verify that `pulse_trains_to_use` has the expected number of pulses per train and near-10 ms spacing.

In [ ]:
plot_pulse_train_structure(
    pulse_trains_to_use,
    max_trains=25,
    expected_isi_s=EXPECTED_ISI_S,
)


## Score Units and Flag Artifact Candidates

Artifact-like units should spike on most pulses, repeat across whole trains, and respond much more on true light trains than on no-light controls.

In [ ]:
artifact_candidates = score_artifact_units(
    units_df=units_df,
    pulse_trains=pulse_trains_to_use,
    light_train_start_times=pulse_train_start_times,
    control_start_times=known_no_light_start_times,
)

artifact_only_units = artifact_candidates[artifact_candidates['artifact_flag']].copy()

summary_cols = [
    'probe',
    'cluster_id',
    'KSlabel',
    'bc_label',
    'brain_region',
    'artifact_score',
    'pulse_response_fraction',
    'near_full_train_fraction',
    'light_train_start_response_fraction',
    'control_start_response_fraction',
    'light_control_response_diff',
    'latency_jitter_ms',
    'response_baseline_ratio',
    'spike_exclusive_to_light_fraction',
]

print(f'Total units scored: {len(artifact_candidates)}')
print(f'Flagged artifact-only candidates: {len(artifact_only_units)}')
artifact_only_units[summary_cols].head(25)


## Review One Candidate

Leave `REVIEW_PROBE` and `REVIEW_CLUSTER_ID` as `None` to inspect the top flagged candidate automatically.

In [ ]:
REVIEW_PROBE = None
REVIEW_CLUSTER_ID = None

review_df = artifact_only_units if len(artifact_only_units) else artifact_candidates

if review_df.empty:
    print('No units available to review.')
else:
    if REVIEW_CLUSTER_ID is None:
        chosen_unit = review_df.iloc[0]
    else:
        mask = review_df['cluster_id'].eq(REVIEW_CLUSTER_ID)
        if REVIEW_PROBE is not None and 'probe' in review_df.columns:
            mask &= review_df['probe'].eq(REVIEW_PROBE)
        if not mask.any():
            raise ValueError('Requested REVIEW_PROBE / REVIEW_CLUSTER_ID not found in review table.')
        chosen_unit = review_df.loc[mask].iloc[0]

    chosen_unit_with_spikes = units_df.loc[
        units_df['cluster_id'].eq(chosen_unit['cluster_id'])
        & units_df['probe'].eq(chosen_unit['probe'])
    ].iloc[0]

    display(pd.DataFrame([chosen_unit[summary_cols].to_dict()]))
    plot_artifact_candidate(chosen_unit_with_spikes, pulse_trains_to_use)
